# Grafana Logs Preprocessing

This notebook handles preprocessing of Grafana logs with the following challenges:
- **Nested JSON structure**: Multiple levels of nested dictionaries (panel, target, datapoints, tags, meta)
- **Variable-length arrays**: Datapoints contain arrays of [value, timestamp] pairs
- **Mixed data types**: Numeric values, strings, and complex objects
- **Prometheus queries**: Complex query expressions that need parsing
- **Multiple panel types**: Different visualization types (gauge, graph, stat, heatmap)
- **Timestamp formats**: Milliseconds in datapoints

## Steps:
1. Load and explore the data
2. Handle nested structures (flatten JSON)
3. Extract datapoints and parse timestamps
4. Handle missing values and outliers
5. Clean and normalize text fields
6. Parse Prometheus queries for semantic information
7. Save preprocessed data

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

print("Libraries imported successfully!")

## 1. Load and Explore Data

In [ ]:
# Load JSONL file
data_path = '../synthetic-log-generator/output/grafana/logs_2024-01-01.jsonl'

logs = []
with open(data_path, 'r') as f:
    for line in f:
        try:
            logs.append(json.loads(line))
        except json.JSONDecodeError as e:
            print(f"Error parsing line: {e}")

print(f"Total logs loaded: {len(logs)}")
print(f"\nSample log entry:")
print(json.dumps(logs[0], indent=2))

In [ ]:
# Explore log structure
print("Top-level keys:")
print(logs[0].keys())

# Check for different types of logs (panels, annotations, alerts)
log_types = {}
for log in logs:
    if 'panel' in log:
        log_types['panel'] = log_types.get('panel', 0) + 1
    if 'type' in log and log.get('type') == 'annotation':
        log_types['annotation'] = log_types.get('annotation', 0) + 1
    if 'state' in log:
        log_types['alert'] = log_types.get('alert', 0) + 1

print(f"\nLog type distribution:")
for log_type, count in log_types.items():
    print(f"  {log_type}: {count}")

## 2. Flatten Nested JSON Structure

In [ ]:
def flatten_log_entry(log):
    """
    Flatten nested Grafana log entry into a flat dictionary.
    Handles different log types: panels, annotations, alerts.
    """
    flattened = {}
    
    # Handle panel logs (most common)
    if 'panel' in log:
        flattened['log_type'] = 'panel'
        flattened['dashboard'] = log.get('dashboard', '')
        
        # Panel information
        panel = log.get('panel', {})
        flattened['panel_id'] = panel.get('id', None)
        flattened['panel_title'] = panel.get('title', '')
        flattened['panel_type'] = panel.get('type', '')
        flattened['datasource'] = panel.get('datasource', '')
        
        # Target information
        target = log.get('target', {})
        flattened['query'] = target.get('expr', '')
        flattened['legend_format'] = target.get('legendFormat', '')
        flattened['ref_id'] = target.get('refId', '')
        
        # Datapoints - extract first value and timestamp
        datapoints = log.get('datapoints', [[None, None]])
        if datapoints and len(datapoints) > 0 and len(datapoints[0]) >= 2:
            flattened['value'] = datapoints[0][0]
            flattened['timestamp_ms'] = datapoints[0][1]
        else:
            flattened['value'] = None
            flattened['timestamp_ms'] = None
        
        # Tags
        tags = log.get('tags', {})
        flattened['service'] = tags.get('service', '')
        flattened['environment'] = tags.get('environment', '')
        flattened['cluster'] = tags.get('cluster', '')
        
        # Meta information
        meta = log.get('meta', {})
        flattened['visualization_type'] = meta.get('preferredVisualisationType', '')
    
    # Handle annotations
    elif 'type' in log and log.get('type') == 'annotation':
        flattened['log_type'] = 'annotation'
        flattened['annotation_id'] = log.get('id', None)
        flattened['dashboard_id'] = log.get('dashboardId', None)
        flattened['title'] = log.get('title', '')
        flattened['text'] = log.get('text', '')
        flattened['timestamp_ms'] = log.get('time', None)
        flattened['tags'] = ','.join(log.get('tags', []))
    
    # Handle alerts
    elif 'state' in log:
        flattened['log_type'] = 'alert'
        flattened['alert_id'] = log.get('id', None)
        flattened['dashboard_id'] = log.get('dashboardId', None)
        flattened['panel_id'] = log.get('panelId', None)
        flattened['alert_name'] = log.get('name', '')
        flattened['state'] = log.get('state', '')
        flattened['state_date'] = log.get('newStateDate', '')
        
        # Extract evaluation data
        eval_data = log.get('evalData', {})
        eval_matches = eval_data.get('evalMatches', [])
        if eval_matches:
            flattened['value'] = eval_matches[0].get('value', None)
            flattened['metric'] = eval_matches[0].get('metric', '')
            tags = eval_matches[0].get('tags', {})
            flattened['service'] = tags.get('service', '')
            flattened['environment'] = tags.get('environment', '')
        
        # Extract threshold info
        eval_info = log.get('evalInfo', {})
        flattened['threshold'] = eval_info.get('threshold', None)
        flattened['condition'] = eval_info.get('condition', '')
    
    return flattened

# Apply flattening to all logs
flattened_logs = [flatten_log_entry(log) for log in logs]
df = pd.DataFrame(flattened_logs)

print(f"Flattened DataFrame shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
df.head()

In [ ]:
# Check data types and missing values
print("Data types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())
print("\nMissing value percentage:")
print((df.isnull().sum() / len(df) * 100).round(2))

## 3. Parse Timestamps and Convert Data Types

In [ ]:
# Convert timestamp from milliseconds to datetime
df['timestamp'] = pd.to_datetime(df['timestamp_ms'], unit='ms', errors='coerce')

# Extract temporal features
df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek
df['day_name'] = df['timestamp'].dt.day_name()
df['minute'] = df['timestamp'].dt.minute
df['second'] = df['timestamp'].dt.second

print("Timestamp statistics:")
print(df['timestamp'].describe())

# Plot temporal distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Hourly distribution
df['hour'].value_counts().sort_index().plot(kind='bar', ax=axes[0])
axes[0].set_title('Log Distribution by Hour')
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Count')

# Log type distribution
df['log_type'].value_counts().plot(kind='bar', ax=axes[1])
axes[1].set_title('Log Type Distribution')
axes[1].set_xlabel('Log Type')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

## 4. Handle Missing Values and Outliers

In [ ]:
# Focus on panel logs for metric analysis
df_panels = df[df['log_type'] == 'panel'].copy()
print(f"Panel logs: {len(df_panels)}")

# Convert value to numeric
df_panels['value'] = pd.to_numeric(df_panels['value'], errors='coerce')

# Analyze value distribution by panel type
print("\nValue statistics by panel title:")
value_stats = df_panels.groupby('panel_title')['value'].describe()
print(value_stats)

# Visualize value distributions
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Overall value distribution
df_panels['value'].hist(bins=50, ax=axes[0, 0])
axes[0, 0].set_title('Overall Value Distribution')
axes[0, 0].set_xlabel('Value')
axes[0, 0].set_ylabel('Frequency')

# Log-scale value distribution
df_panels[df_panels['value'] > 0]['value'].apply(np.log10).hist(bins=50, ax=axes[0, 1])
axes[0, 1].set_title('Log10 Value Distribution')
axes[0, 1].set_xlabel('Log10(Value)')
axes[0, 1].set_ylabel('Frequency')

# Boxplot by panel type
top_panels = df_panels['panel_title'].value_counts().head(10).index
df_panels[df_panels['panel_title'].isin(top_panels)].boxplot(
    column='value', by='panel_title', ax=axes[1, 0], rot=45
)
axes[1, 0].set_title('Value Distribution by Panel Title')
axes[1, 0].set_xlabel('Panel Title')
axes[1, 0].set_ylabel('Value')

# Service distribution
df_panels['service'].value_counts().head(10).plot(kind='barh', ax=axes[1, 1])
axes[1, 1].set_title('Top 10 Services')
axes[1, 1].set_xlabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# Detect and handle outliers using IQR method per panel type
def remove_outliers_iqr(group, column='value', threshold=3.0):
    """
    Remove outliers using IQR method for a specific group.
    """
    Q1 = group[column].quantile(0.25)
    Q3 = group[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - threshold * IQR
    upper_bound = Q3 + threshold * IQR
    
    return group[(group[column] >= lower_bound) & (group[column] <= upper_bound)]

# Store original count
original_count = len(df_panels)

# Remove outliers per panel type
df_panels_clean = df_panels.groupby('panel_title', group_keys=False).apply(
    lambda x: remove_outliers_iqr(x, 'value', threshold=3.0)
)

print(f"Original panel logs: {original_count}")
print(f"After outlier removal: {len(df_panels_clean)}")
print(f"Removed: {original_count - len(df_panels_clean)} ({(original_count - len(df_panels_clean)) / original_count * 100:.2f}%)")

## 5. Parse Prometheus Queries for Semantic Features

In [ ]:
import re

def extract_query_features(query):
    """
    Extract semantic features from Prometheus queries.
    """
    features = {}
    
    if pd.isna(query) or query == '':
        return features
    
    # Extract metric name
    metric_match = re.match(r'^([a-zA-Z_][a-zA-Z0-9_]*)', query)
    features['metric_name'] = metric_match.group(1) if metric_match else ''
    
    # Check for aggregation functions
    features['has_rate'] = 'rate(' in query
    features['has_histogram_quantile'] = 'histogram_quantile' in query
    features['has_sum'] = 'sum(' in query or '_sum' in query
    features['has_count'] = 'count(' in query or '_count' in query
    features['has_avg'] = 'avg(' in query
    features['has_max'] = 'max(' in query
    features['has_min'] = 'min(' in query
    
    # Extract time window
    time_window_match = re.search(r'\[(\d+)([smhd])\]', query)
    if time_window_match:
        features['time_window_value'] = int(time_window_match.group(1))
        features['time_window_unit'] = time_window_match.group(2)
    else:
        features['time_window_value'] = 0
        features['time_window_unit'] = ''
    
    # Check for specific metric types
    features['is_cpu_metric'] = 'cpu' in query.lower()
    features['is_memory_metric'] = 'memory' in query.lower()
    features['is_network_metric'] = 'network' in query.lower()
    features['is_http_metric'] = 'http' in query.lower()
    features['is_database_metric'] = any(db in query.lower() for db in ['postgresql', 'mysql', 'redis', 'database'])
    features['is_jvm_metric'] = 'jvm' in query.lower()
    
    # Count complexity (number of operators)
    features['query_complexity'] = query.count('{') + query.count('[') + query.count('(')
    
    return features

# Apply query feature extraction
query_features = df_panels_clean['query'].apply(extract_query_features)
query_features_df = pd.DataFrame(query_features.tolist())

# Merge with main dataframe
df_panels_clean = pd.concat([df_panels_clean.reset_index(drop=True), query_features_df], axis=1)

print("Query feature extraction complete!")
print(f"\nNew columns: {list(query_features_df.columns)}")
print("\nQuery feature statistics:")
print(query_features_df.describe())

## 6. Text Feature Cleaning and Encoding

In [ ]:
# Clean text fields
text_columns = ['dashboard', 'panel_title', 'panel_type', 'datasource', 'service', 'environment']

for col in text_columns:
    if col in df_panels_clean.columns:
        # Fill missing values
        df_panels_clean[col] = df_panels_clean[col].fillna('unknown')
        # Strip whitespace
        df_panels_clean[col] = df_panels_clean[col].str.strip()
        # Convert to lowercase
        df_panels_clean[col] = df_panels_clean[col].str.lower()

print("Text cleaning complete!")
print("\nUnique values per categorical column:")
for col in text_columns:
    if col in df_panels_clean.columns:
        print(f"  {col}: {df_panels_clean[col].nunique()}")

## 7. Summary Statistics and Data Quality Report

In [ ]:
print("=" * 80)
print("DATA QUALITY REPORT")
print("=" * 80)

print(f"\n1. Dataset Overview:")
print(f"   - Total records: {len(df_panels_clean):,}")
print(f"   - Total columns: {len(df_panels_clean.columns)}")
print(f"   - Memory usage: {df_panels_clean.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print(f"\n2. Temporal Coverage:")
print(f"   - Start time: {df_panels_clean['timestamp'].min()}")
print(f"   - End time: {df_panels_clean['timestamp'].max()}")
print(f"   - Duration: {df_panels_clean['timestamp'].max() - df_panels_clean['timestamp'].min()}")

print(f"\n3. Data Completeness:")
completeness = (1 - df_panels_clean.isnull().sum() / len(df_panels_clean)) * 100
for col in df_panels_clean.columns:
    if completeness[col] < 100:
        print(f"   - {col}: {completeness[col]:.2f}% complete")

print(f"\n4. Value Statistics:")
print(f"   - Mean: {df_panels_clean['value'].mean():.2f}")
print(f"   - Median: {df_panels_clean['value'].median():.2f}")
print(f"   - Std Dev: {df_panels_clean['value'].std():.2f}")
print(f"   - Min: {df_panels_clean['value'].min():.2f}")
print(f"   - Max: {df_panels_clean['value'].max():.2f}")

print(f"\n5. Categorical Distribution:")
print(f"   - Unique dashboards: {df_panels_clean['dashboard'].nunique()}")
print(f"   - Unique panels: {df_panels_clean['panel_title'].nunique()}")
print(f"   - Unique services: {df_panels_clean['service'].nunique()}")
print(f"   - Unique panel types: {df_panels_clean['panel_type'].nunique()}")

print("\n" + "=" * 80)

## 8. Save Preprocessed Data

In [ ]:
# Save preprocessed data
output_path = 'preprocessed_grafana_logs.csv'
df_panels_clean.to_csv(output_path, index=False)
print(f"Preprocessed data saved to: {output_path}")

# Also save a pickle for faster loading
pickle_path = 'preprocessed_grafana_logs.pkl'
df_panels_clean.to_pickle(pickle_path)
print(f"Preprocessed data (pickle) saved to: {pickle_path}")

print(f"\nFinal dataset shape: {df_panels_clean.shape}")
print(f"Columns: {list(df_panels_clean.columns)}")

In [ ]:
# Display sample of preprocessed data
df_panels_clean.head(10)